# Treinamento do Style-Bert-VITS2 (ver 2.7.0) no Google Colab

Você pode treinar o Style-Bert-VITS2 no Google Colab.

Neste notebook, para uso normal, uma pasta `Style-Bert-VITS2` será criada no seu Google Drive e o trabalho será realizado dentro dela. Nenhuma outra pasta será tocada.
Se você não usar o Google Drive, especifique o caminho apropriado na configuração inicial.

## Fluxo

### Quando quiser começar o treinamento do zero
Basta executar de cima para baixo. Os arquivos necessários para síntese de voz serão salvos em `Style-Bert-VITS2/model_assets/` no Google Drive. Além disso, o progresso intermediário também é salvo em `Style-Bert-VITS2/Data/`, para que você possa interromper o treinamento ou retomá-lo do meio.

### Quando quiser retomar o treinamento do meio
Execute 0 e 1, pule o pré-processamento 3 e comece do 4. A divisão de estilo 5 deve ser feita após o término do treinamento, se necessário.


## 0. Configuração do Ambiente

Construa o ambiente Style-Bert-VITS2 no Colab. Verifique se o runtime é um backend de GPU como T4 e execute.

**Atenção**: Após executar esta célula, avisos como "A sessão falhou" ou "A sessão falhou por motivo desconhecido" aparecerão, mas **ignore e prossiga**. (Isso ocorre porque `exit()` é chamado para reiniciar o runtime e forçar numpy<2.)

In [8]:
import os

os.environ["PATH"] += ":/root/.cargo/bin"

!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone https://github.com/neto007/Style-Bert-VITS2.git
%cd Style-Bert-VITS2/
!uv pip install --system -r requirements-colab.txt --no-progress
!python initialize.py --skip_default_models

exit()

Using Python 3.12.12 environment at: /usr
Audited 26 packages in 101ms


In [4]:
# Execute isto se você usar o Google Drive.

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

os.environ["PATH"] += ":/root/.cargo/bin"

#!curl -LsSf https://astral.sh/uv/install.sh | sh
#!git clone https://github.com/neto007/Style-Bert-VITS2.git
%cd Style-Bert-VITS2/
!uv pip install --system -r requirements-colab.txt --no-progress
!python initialize.py --skip_default_models

exit()

/content/Style-Bert-VITS2
Using Python 3.12.12 environment at: /usr
Audited 26 packages in 101ms


## 1. Configuração Inicial

Especifique o nome do diretório para salvar o treinamento e seus resultados.
Se for Google Drive, execute como está; se quiser personalizar, altere e execute.

In [5]:
# Mover para o diretório de trabalho
%cd /content/Style-Bert-VITS2/

# Diretório onde arquivos necessários para treinamento e progresso intermediário são salvos
dataset_root = "/content/drive/MyDrive/Style-Bert-VITS2/Data"

# Diretório onde os resultados do treinamento (arquivos necessários para síntese de voz) são salvos
assets_root = "/content/drive/MyDrive/Style-Bert-VITS2/model_assets"

import yaml


with open("configs/paths.yml", "w", encoding="utf-8") as f:
    yaml.dump({"dataset_root": dataset_root, "assets_root": assets_root}, f)

/content/Style-Bert-VITS2


## 2. Preparação de Dados para Treinamento

Se você já tiver arquivos de áudio (cerca de 2-12 segundos por arquivo) e seus dados de transcrição, execute 2.2; caso contrário, execute 2.1.

### 2.1 Criação de Dataset a partir de Arquivos de Áudio (pode pular se já tiver)

Se você não tiver um dataset de arquivos de áudio (cerca de 2-12 segundos por arquivo) e suas transcrições, pode criar um dataset a partir de arquivos de áudio (em japonês) seguindo os passos abaixo. Coloque os arquivos de áudio (formatos normais como wav ou mp3, um ou vários arquivos) na pasta `Style-Bert-VITS2/inputs/` no Google Drive e execute o comando abaixo para criar o dataset e colocá-lo automaticamente no local correto.

**A partir da Ver 2.5 de 02/06/2024**, se você criar 2 ou mais subpastas na pasta `inputs/` e distribuir os arquivos de áudio de acordo com o estilo, os estilos correspondentes aos subdiretórios serão criados automaticamente durante o treinamento. Se apenas o estilo padrão for suficiente ou se você for criar estilos manualmente depois, pode colocá-los diretamente em `inputs/`.

In [18]:
# Diretório para colocar arquivos de áudio de origem (formato wav)
input_dir = "/content/drive/MyDrive/finetuneArtedaGuerra/Arte/wavs"
# Digite o nome do modelo (nome do falante)
model_name = "artedaguerra"

# Exemplo de frase como você quer que seja transcrito (pontuação, risadas, nomes próprios, etc.)
initial_prompt = "Olá. Tudo bem? Hehe, eu... estou bem!"

!python slice.py -i {input_dir} --model_name {model_name}
!python transcribe.py --model_name {model_name} --initial_prompt "{initial_prompt}" --use_hf_whisper --hf_repo_id openai/whisper-large-v2

11-26 20:13:57 |  INFO  | slice.py:132 | Found 802 audio files.
11-26 20:13:57 |WARNING | slice.py:134 | Output directory /content/drive/MyDrive/Style-Bert-VITS2/Data/artedaguerra/raw already exists, deleting...
Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master
  0%|          | 0/802 [00:00<?, ?it/s]Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master
Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master
Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master
100%|##########| 802/802 [01:17<00:00, 10.28it/s]
11-26 20:15:17 |  INFO  | slice.py:226 | Slice done! Total time: 54.53 min, 704 files.
11-26 20:15:20 |  INFO  | transcribe.py:157 | Found 694 WAV files
11-26 20:15:20 |  INFO  | transcribe.py:206 | Loading HF Whisper model (openai/whisper-large-v2)
  0%|          | 0/694 [00:00<?, ?it/s]2025-11-26 20:15:26.401367: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: A

In [ ]:
!pip install torch==2.0.1 torchaudio==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118


Se tiver sucesso, prossiga para o passo 3

### 2.2 Se você já tiver arquivos de áudio e dados de transcrição

Coloque o dataset adequadamente seguindo as instruções.

Execute a próxima célula para criar a pasta onde colocar os dados de treinamento (`dataset_root` configurado em 1).

In [ ]:
import os

os.makedirs(dataset_root, exist_ok=True)

Primeiro, prepare os dados de áudio e o texto de transcrição.

Organize-os da seguinte maneira:
```
├── Data/
│   ├── {nome_do_modelo}
│   │   ├── esd.list
│   │   ├── raw/
│   │   │   ├── foo.wav
│   │   │   ├── bar.mp3
│   │   │   ├── style1/
│   │   │   │   ├── baz.wav
│   │   │   │   ├── qux.wav
│   │   │   ├── style2/
│   │   │   │   ├── corge.wav
│   │   │   │   ├── grault.wav
...
```

### Como organizar
- Se organizado como acima, além do estilo padrão, os estilos `style1` e `style2` serão criados automaticamente a partir dos arquivos de áudio dentro das pastas `style1/` e `style2/` (incluindo subpastas).
- Se não precisar criar estilos específicos, ou se for criar estilos usando a função de classificação de estilo, coloque tudo diretamente na pasta `raw/`. Se o número de subdiretórios em `raw/` for 0 ou 1, apenas o estilo padrão será criado.
- O formato do arquivo de áudio suporta muitos formatos como mp3 além do formato wav.

### Arquivo de transcrição `esd.list`

No arquivo `Data/{nome_do_modelo}/esd.list`, descreva as informações de cada arquivo de áudio no seguinte formato:


```
path/to/audio.wav(mesmo se não for wav)|{nome_do_falante}|{ID_do_idioma, ZH, JP ou EN}|{texto_de_transcrição}
```

- Aqui, o primeiro `path/to/audio.wav` é o caminho relativo a partir de `raw/`. Ou seja, para `raw/foo.wav` é `foo.wav`, e para `raw/style1/bar.wav` é `style1/bar.wav`.
- Mesmo que a extensão não seja wav, escreva `wav` no `esd.list`. Por exemplo, para `raw/bar.mp3`, escreva `bar.wav`.


Exemplo:
```
foo.wav|hanako|JP|Olá, como vai?
bar.wav|taro|JP|Sim, estou ouvindo... Precisa de algo?
style1/baz.wav|hanako|JP|O tempo está bom hoje, não é?
style1/qux.wav|taro|JP|Sim, é verdade.
...
english_teacher.wav|Mary|EN|How are you? I'm fine, thank you, and you?
...
```
Claro, um conjunto de dados de um único falante japonês também é aceitável.

## 3. Pré-processamento do Treinamento

Em seguida, realizamos o pré-processamento do treinamento. Especifique os parâmetros necessários aqui. Insira as configurações na próxima célula e execute. Especifique `True` ou `False` para opções de "sim/não".

In [ ]:
# Nome da pasta criada acima `Data/{model_name}/`
model_name = "artedaguerra"

# Usar JP-Extra (versão especializada em japonês)? Melhora a capacidade em japonês, mas perde inglês e chinês.
use_jp_extra = True

# Tamanho do batch de treinamento. Ajuste de acordo com a VRAM disponível.
batch_size = 4

# Número de épocas de treinamento (quantas vezes percorrer o dataset).
# 100 pode ser demais, mas mais épocas podem melhorar a qualidade.
epochs = 100

# Frequência de salvamento. A cada quantos passos salvar o modelo. Se não souber, deixe o padrão.
save_every_steps = 1000

# Normalizar o volume dos arquivos de áudio?
normalize = False

# Remover silêncio no início e fim dos arquivos de áudio?
trim = False

# O que fazer em caso de erro de leitura.
# "raise" interrompe após pré-processamento de texto, "skip" não usa linhas ilegíveis, "use" força o uso.
yomi_error = "skip"

Após executar a célula acima, execute a próxima célula para realizar o pré-processamento do treinamento.

In [ ]:
from gradio_tabs.train import preprocess_all
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker


pyopenjtalk_worker.initialize_worker()

preprocess_all(
    model_name=model_name,
    batch_size=batch_size,
    epochs=epochs,
    save_every_steps=save_every_steps,
    num_processes=2,
    normalize=normalize,
    trim=trim,
    freeze_EN_bert=False,
    freeze_JP_bert=False,
    freeze_ZH_bert=False,
    freeze_style=False,
    freeze_decoder=False,
    use_jp_extra=use_jp_extra,
    val_per_lang=0,
    log_interval=200,
    yomi_error=yomi_error,
)

## 4. Treinamento

Se o pré-processamento terminar normalmente, prossiga para o treinamento. Execute a próxima célula para iniciar.

Os resultados do treinamento serão salvos na pasta `Style-Bert-VITS2/Data/{nome_do_modelo}/model_assets/` no Google Drive no intervalo `save_every_steps` especificado acima.

Baixe esta pasta e sobrescreva a pasta `model_assets` do seu Style-Bert-VITS2 local para usar os resultados do treinamento.

In [ ]:
# Digite o nome do modelo definido acima. Se estiver retomando o treinamento, digite o nome da pasta onde o modelo está salvo.
model_name = "your_model_name"


import yaml
from gradio_tabs.train import get_path

paths = get_path(model_name)
dataset_path = str(paths.dataset_path)
config_path = str(paths.config_path)

with open("default_config.yml", "r", encoding="utf-8") as f:
    yml_data = yaml.safe_load(f)
yml_data["model_name"] = model_name
with open("config.yml", "w", encoding="utf-8") as f:
    yaml.dump(yml_data, f, allow_unicode=True)

In [ ]:
# Se "usar" a versão especializada em japonês
!python train_ms_jp_extra.py --config {config_path} --model {dataset_path} --assets_root {assets_root}

In [ ]:
# Se "não usar" a versão especializada em japonês
!python train_ms.py --config {config_path} --model {dataset_path} --assets_root {assets_root}

In [ ]:
# Testar resultados, mesclar e dividir estilos aqui
!python app.py --share

In [ ]:
# Para conversão ONNX, especifique o arquivo safetensors que deseja converter e execute esta célula.
!python convert_onnx.py --model "Data/your_model/your_model_e100_s10000.safetensors"